# 실습 2 · 파트 2 — 내 파노라마 만들기

**내 사진에서 대응을 자동으로 찾고(SIFT + ratio test), 호모그래피를 직접 구현해(DLT) 워핑·블렌딩한다**

| 노트북 | 내용 |
|---|---|
| 파트 1 | 내 카메라 캘리브레이션 — 촬영 규칙 · 보드 촬영 · K와 σ 확인 |
| **파트 2~3** (이 파일) | **매칭** `빈칸 ①` · **호모그래피 DLT** `빈칸 ②` · RANSAC · 워핑 · **블렌딩** `빈칸 ③` → 내 파노라마 완성 · 모듈화 |
| 파트 4 (보너스) | 시간을 잇는 호모그래피 — "Dear Photograph" (클릭 대응 → 내 DLT) |

**오늘의 빈칸 세 개가 모두 이 노트북에 있습니다** — ① ratio 임계값 (1줄) · ② DLT의 A 행렬 2행
(오늘의 메인 구현) · ③ 블렌딩 가중치 (1줄).
빈칸이 비어 있어도 노트북은 `solutions.py`로 **임시 진행**되지만, 화면에 ⚠ 경고가 남습니다.
빈칸을 채우고 해당 셀부터 재실행해 ⚠를 지우는 것이 목표입니다.

이 노트북은 **단독 실행 가능**합니다. 파트 1을 먼저 돌려 `out/my_params.json` 이 있으면
**내 K로 왜곡 보정**까지 적용하고, 없으면 보정 없이 진행합니다.
**폰 촬영이 여의치 않다면** — `my_pano/` 를 비워 두면 대체 데이터(`data/hyu_*.jpeg`)로 자동 진행됩니다.

In [ ]:
# ── 셀 1 · 환경 준비 (+ 파트 1 캘리브 불러오기) ──────────────────
import cv2, json, math, time, os, glob
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageOps

import matplotlib.font_manager as fm
for cand in ['Malgun Gothic', 'AppleGothic', 'NanumGothic',
             'Noto Sans CJK KR', 'Noto Sans CJK JP', 'Noto Sans KR']:
    if any(cand == f.name for f in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = cand; break
plt.rcParams['axes.unicode_minus'] = False

# iPhone 사진은 확장자가 .jpg/.jpeg여도 내용이 HEIC인 경우가 많습니다 (전송 시 흔함)
try:
    import pillow_heif; pillow_heif.register_heif_opener()
except ImportError:
    print('⚠ pillow-heif 미설치 — HEIC 사진은 열리지 않습니다(UnidentifiedImageError).')
    print('  터미널에서  pip install pillow-heif  실행 후 커널 재시작하세요.')

BASE = Path('.'); OUT = BASE / 'out'; OUT.mkdir(exist_ok=True)
PANO_DIR = BASE / 'my_pano'      # ← 내 폰으로 찍은 파노라마용 사진 3~4장
MAX_SIDE = 1600                  # 파노라마 처리 해상도 (긴 변) — 폰 원본은 이 크기로 축소
                                 # (예: 5712x4284 → 1600x1200). K는 같은 배율로 스케일해 사용

def load_image(path, max_side=None):
    # EXIF 회전을 반영해 로드 (폰 사진 필수) → BGR, 필요 시 축소
    im = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    img = cv2.cvtColor(np.array(im), cv2.COLOR_RGB2BGR)
    if max_side and max(img.shape[:2]) > max_side:
        s = max_side / max(img.shape[:2])
        img = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    return img

def show(img, title='', figsize=(9, 5.4)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title(title); plt.axis('off'); plt.show()

IMG_EXT = ('.jpg', '.jpeg', '.png', '.heic', '.heif')

def list_images(folder):
    # 확장자 대소문자 무관 · HEIC 포함 · 맥의 '._' 숨김 파일 제외
    if not folder.exists():
        return []
    return sorted(p for p in folder.iterdir()
                  if p.suffix.lower() in IMG_EXT and not p.name.startswith('.'))

# 파트 1의 산출물이 있으면 불러온다 (없으면 MY=None → 왜곡 보정 생략)
MY = None
_p = OUT / 'my_params.json'
if _p.exists():
    MY = json.load(open(_p, encoding='utf-8'))
    MY['K'] = np.array(MY['K'], float); MY['dist'] = np.array(MY['dist'], float)
    print(f"파트 1 캘리브 로드 — fx {MY['K'][0, 0]:.1f} ± {MY['sigma_fx']:.2f} px, "
          f"RMS {MY['rms']:.3f} px ({MY['n']}장)")
else:
    print('ℹ out/my_params.json 없음 — 파트 1을 실행하면 내 K로 왜곡 보정까지 적용됩니다 (지금은 생략).')

print('OpenCV', cv2.__version__)
print('파노라마 사진:', len(list_images(PANO_DIR)), '장 /', PANO_DIR)

---
## Part 2 · 파노라마 촬영과 매칭

**파노라마 촬영 규칙** — 발을 고정하고 **제자리 회전만** (이론 3의 순수 회전 조건).
- 강의실 근처 풍경으로 **3장**, 인접 사진 **겹침 30~40%**
- AE 잠금 유지(밝기 차이 최소화), 같은 렌즈·같은 방향
- 가까운 물체(1 m 이내)가 크게 들어가면 시차 → 고스트 — 일부러 한 장 넣어 보는 것도 실험!
- 전송 → `my_pano/` (왼쪽부터 순서대로 파일명 정렬되게: `p0.jpg, p1.jpg, p2.jpg` 권장)

폴더가 비어 있으면 자동으로 `data/hyu_*.jpeg` (한양대 캠퍼스, 이전 기수 촬영본)로 진행합니다.

In [ ]:
# ── 셀 3 · 파노라마 입력 로드 (축소 + 선택: 내 K로 왜곡 보정) ───────
pano_files = list_images(PANO_DIR) or sorted((BASE/'data').glob('hyu_*.jpeg'))
IS_MINE = pano_files[0].parent.name == 'my_pano'
SRC_NOTE = '내 촬영본' if IS_MINE else '대체 데이터(hyu)'

# 폰 원본은 수천만 화소 — SIFT/매칭이 느릴 뿐 아니라 기본 파라미터(옥타브·임계값)와도
# 잘 안 맞습니다. 긴 변을 MAX_SIDE(HD급)로 줄여 처리합니다.
imgs, raw_sizes = [], []
for f in pano_files:
    im = load_image(f)                                   # 원본 (EXIF 회전 반영)
    raw_sizes.append((im.shape[1], im.shape[0]))
    s = min(1.0, MAX_SIDE / max(im.shape[:2]))
    if s < 1.0:
        im = cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    imgs.append(im)
assert len(imgs) >= 2, '파노라마에는 최소 2장이 필요합니다'
assert len({im.shape for im in imgs}) == 1, \
    f'해상도가 섞여 있습니다 {[im.shape[1::-1] for im in imgs]} — 가로/세로 방향을 통일해 다시 찍으세요'

RW, RH = raw_sizes[0]                                     # 원본 해상도
PW, PH = imgs[0].shape[1], imgs[0].shape[0]               # 처리 해상도
SCALE = PW / RW                                           # 축소 배율 (K 스케일에 그대로 사용)
print(f'{SRC_NOTE} {len(imgs)}장 — 원본 {RW}x{RH} → 처리 {PW}x{PH} (x{SCALE:.3f})')

# 왜곡 보정은 '내 사진 + 내 캘리브'가 짝일 때만. 대체 데이터(hyu)는 다른 카메라로 찍은
# 사진이라 내 K를 적용하면 화각·왜곡이 어긋나 영상이 크게 잘립니다.
UNDISTORT = IS_MINE and MY is not None       # (끄고 비교 가능)
if UNDISTORT:
    CW_, CH_ = MY['size']                                 # 캘리브 때의 해상도
    sx, sy = PW / CW_, PH / CH_                           # 캘리브 → 처리 해상도 배율
    if abs(sx - sy) / max(sx, sy) > 0.02:
        print(f'⚠ 캘리브 {CW_}x{CH_} 와 파노라마 {PW}x{PH} 의 화면비가 다릅니다 '
              '(가로/세로 방향 불일치?) — 이번에는 왜곡 보정을 건너뜁니다.')
        UNDISTORT = False
    else:
        # 축소는 픽셀 좌표의 단순 스케일이므로 K도 같은 배율로: fx,fy,cx,cy 전부 x s.
        # 왜곡계수(k1,k2,p1,p2,k3)는 정규화 좌표 기반이라 해상도와 무관 — 그대로 사용.
        Ks = np.diag([sx, sy, 1.0]) @ MY['K']
        newK, _ = cv2.getOptimalNewCameraMatrix(Ks, MY['dist'], (PW, PH), 0)
        imgs = [cv2.undistort(im, Ks, MY['dist'], None, newK) for im in imgs]
        print(f'왜곡 보정 적용 — 캘리브 {CW_}x{CH_} 의 K를 x{sx:.3f} 스케일해 사용 '
              f'(fx {MY["K"][0, 0]:.0f} → {Ks[0, 0]:.1f}), 왜곡계수는 그대로')
if not UNDISTORT:
    print('왜곡 보정 생략 —',
          '대체 데이터(hyu)는 다른 카메라 촬영본이라 내 K를 적용하지 않습니다 (적용 시 화각이 어긋나 잘림)'
          if not IS_MINE else 'MY 없음 또는 UNDISTORT=False')

show(np.hstack([cv2.resize(im, None, fx=0.33, fy=0.33) for im in imgs]),
     f'입력 {len(imgs)}장 — {SRC_NOTE} · {PW}x{PH}' + (' · 왜곡 보정됨' if UNDISTORT else ''))


### Step 1–2 · 특징점 검출과 매칭 — `빈칸 ①`

SIFT가 "대응 자동 공급 장치" 역할을 합니다 (어제는 보드 코너가 하던 일).
`knnMatch(k=2)`로 최근접·차근접을 받아 **ratio test**로 모호한 매칭을 버립니다.

> **빈칸 ①** — `RATIO` 값을 정하세요 (권장 0.75). 정한 뒤 0.9와 0.5로도 바꿔 실행해
> 매칭 수가 어떻게 변하는지 관찰해 보세요. RANSAC의 부담(인라이어 비율 w)과 직결됩니다.

**몇 장이든 됩니다** — `my_pano/`에 넣은 사진 전부(비어 있으면 `data/hyu_*.jpeg` 3장)를 잇습니다.
단, 매칭은 **이웃끼리만** 합니다: 파노라마의 양 끝 영상은 서로 겹치지 않아 직접 매칭이 불가능하기
때문입니다. 이웃 H들을 곱해 기준 영상까지 연결하는 것이 다음 셀의 **체인**입니다.
(그래서 파일명 정렬 순서 = 촬영 순서 가 중요합니다: `p0, p1, p2 …`)

In [ ]:
# ── 셀 4 · SIFT + ratio test (이웃끼리 매칭) ────────────────────
RATIO = None        # TODO ①: 0.5 ~ 0.9 사이 값 하나 (권장 0.75)

if RATIO is None:
    from solutions import DEFAULT_RATIO as RATIO
    print(f'⚠ 빈칸 ① 미완성 — solutions의 {RATIO}로 임시 진행 (채운 뒤 이 셀부터 재실행)')
else:
    print(f'빈칸 ① 자기 값 사용: RATIO = {RATIO}')

try:
    FEAT = cv2.SIFT_create(nfeatures=4000)
    NORM = cv2.NORM_L2
except AttributeError:
    FEAT = cv2.ORB_create(nfeatures=4000); NORM = cv2.NORM_HAMMING
    print('SIFT 불가 → ORB 대체')

grays = [cv2.cvtColor(im, cv2.COLOR_BGR2GRAY) for im in imgs]
kps, descs = zip(*[FEAT.detectAndCompute(g, None) for g in grays])
print('특징점/장:', [len(k) for k in kps])

bf = cv2.BFMatcher(NORM)
def match_pair(i, j):
    # 영상 i → j 의 ratio-test 통과 매칭 좌표 (src=i, dst=j)
    raw = bf.knnMatch(descs[i], descs[j], k=2)
    good = [m for m, n in raw if m.distance < RATIO * n.distance]
    src = np.float32([kps[i][m.queryIdx].pt for m in good])
    dst = np.float32([kps[j][m.trainIdx].pt for m in good])
    return src, dst, good

# N장을 전부 잇기 위해 '이웃끼리만' 매칭한다 (0-1, 1-2, 2-3, …).
# 멀리 떨어진 영상은 겹침이 없어 직접 매칭이 불가능 — 이웃 H를 곱해 연결한다(셀 6).
REF = len(imgs) // 2                              # 가운데 영상을 기준으로
ADJ = [(i, i + 1) for i in range(len(imgs) - 1)]
matches = {(i, j): match_pair(i, j) for i, j in ADJ}
for (i, j), (s, d, g) in matches.items():
    print(f'영상 {i} ↔ {j}: 매칭 {len(g)}개')
assert all(len(g) >= 8 for _, _, g in matches.values()), \
    '이웃 매칭이 부족합니다 — 파일명 순서(왼쪽→오른쪽)와 겹침 30~40%를 확인하세요'
print(f'입력 {len(imgs)}장 · 이웃 쌍 {len(ADJ)}개 · 기준 영상 {REF}')

rows = []
for (i, j), (s, d, g) in matches.items():
    v = cv2.drawMatches(imgs[i], kps[i], imgs[j], kps[j], g[:60], None,
                        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    rows.append(cv2.resize(v, (1600, int(v.shape[0] * 1600 / v.shape[1]))))
vis = np.vstack(rows)
show(vis, f'이웃 쌍 {len(rows)}개의 매칭 상위 60 — 평행한 실선 무리 + 어긋난 소수(오매칭)',
     (11, 3.2 * len(rows)))
cv2.imwrite(str(OUT / 'matches.png'), vis)


---
## Part 3 · 호모그래피 직접 구현
### Step 3 · DLT — `빈칸 ②` (오늘의 메인 구현)

오전에 세 번 만난 패턴을 이제 손으로 씁니다. 대응 $(x,y)\to(x',y')$ 하나는
$x' \times Hx = 0$ 에서 **두 개의 선형 방정식**을 만듭니다 ($w=w'=1$):

$$A_i = \begin{pmatrix} 0 & 0 & 0 & -x & -y & -1 & \;y'x & y'y & y' \\ x & y & 1 & 0 & 0 & 0 & -x'x & -x'y & -x' \end{pmatrix}$$

정규화(Hartley)와 SVD는 제공됩니다 — 여러분이 채울 것은 **이 두 행**뿐입니다.
막히면 `from solutions import build_A_rows`가 자동으로 대신하지만, ⚠가 남습니다.

In [11]:
# ── 셀 5 · DLT 구현 (빈칸 ②) ───────────────────────────────────
def build_A_rows(x, y, xp, yp):
    """대응 (x,y)→(x',y') 하나가 만드는 A의 두 행(각 길이 9)을 반환.
    위 수식의 row1, row2 를 리스트로 작성하세요."""
    # TODO ②: 아래 두 줄을 완성 (각각 원소 9개)
    # row1 = [ ... ]
    # row2 = [ ... ]
    raise NotImplementedError('빈칸 ②: A 행렬 2행을 작성하세요')

try:
    _r1, _r2 = build_A_rows(1.0, 2.0, 3.0, 4.0)
    assert len(_r1) == len(_r2) == 9
    A_ROWS = build_A_rows
    print('빈칸 ② 자기 구현 사용 ✅')
except NotImplementedError:
    from solutions import build_A_rows as A_ROWS
    print('⚠ 빈칸 ② 미완성 — solutions로 임시 진행 (채운 뒤 이 셀부터 재실행)')

def _normalize(pts):
    """무게중심 → 원점, 평균 거리 → √2 (Hartley). 반환: 정규화 좌표, 변환 T"""
    c = pts.mean(axis=0)
    d = np.linalg.norm(pts - c, axis=1).mean()
    s = math.sqrt(2) / max(d, 1e-12)
    T = np.array([[s, 0, -s * c[0]], [0, s, -s * c[1]], [0, 0, 1]])
    ph = np.hstack([pts, np.ones((len(pts), 1))]) @ T.T
    return ph[:, :2], T

def estimate_H_dlt(src, dst):
    """정규화 DLT: src→dst 호모그래피 (src, dst: (N,2), N≥4)"""
    sn, Ts = _normalize(np.asarray(src, float))
    dn, Td = _normalize(np.asarray(dst, float))
    A = []
    for (x, y), (xp, yp) in zip(sn, dn):
        r1, r2 = A_ROWS(x, y, xp, yp)
        A.append(r1); A.append(r2)
    _, _, Vt = np.linalg.svd(np.asarray(A))
    Hn = Vt[-1].reshape(3, 3)
    H = np.linalg.inv(Td) @ Hn @ Ts          # 역변환으로 원좌표 복원
    return H / H[2, 2]
print('estimate_H_dlt 준비 완료')

⚠ 빈칸 ② 미완성 — solutions로 임시 진행 (채운 뒤 이 셀부터 재실행)
estimate_H_dlt 준비 완료


### 검증 게이트 — 내 DLT vs OpenCV

RANSAC(제공: `cv2.findHomography`)으로 얻은 **인라이어만** 넣어, 내 DLT와 OpenCV의 H를
같은 잣대(대칭 전송 오차)로 비교합니다. 부호 실수는 여기서 바로 드러납니다.

In [ ]:
# ── 셀 6 · RANSAC(제공) + 검증 게이트 → 체인으로 연결 ──────────
def sym_err(H, src, dst):
    sh = np.hstack([src, np.ones((len(src), 1))]); dh = np.hstack([dst, np.ones((len(dst), 1))])
    f = sh @ H.T; f = f[:, :2] / f[:, 2:3]
    b = dh @ np.linalg.inv(H).T; b = b[:, :2] / b[:, 2:3]
    return float(np.mean(np.linalg.norm(f - dst, axis=1) + np.linalg.norm(b - src, axis=1)))

H_cv, H_mine, inl_masks = {}, {}, {}          # 키: 이웃 쌍 (i, j)
for (i, j), (src, dst, _) in matches.items():
    Hcv, mask = cv2.findHomography(src, dst, cv2.RANSAC, 3.0)
    m = mask.ravel().astype(bool)
    Hme = estimate_H_dlt(src[m], dst[m])
    H_cv[(i, j)], H_mine[(i, j)], inl_masks[(i, j)] = Hcv, Hme, m
    e_cv, e_me = sym_err(Hcv, src[m], dst[m]), sym_err(Hme, src[m], dst[m])
    print(f'영상 {i}→{j}: 인라이어 {m.sum()}/{len(m)} ({m.mean()*100:.0f}%) | '
          f'대칭 전송 오차 — OpenCV {e_cv:.2f}px vs 내 DLT {e_me:.2f}px')
    if abs(e_me - e_cv) < 1.0:
        print('   ✅ 검증 게이트 통과 — 내 DLT가 OpenCV와 동급')
    else:
        print('   ❌ 게이트 미통과 — A 행렬 두 행의 부호·순서를 다시 확인하세요')

USE_MINE = True     # 이후 파이프라인에 내 DLT(H_mine) 사용 (False → OpenCV)
H_adj = H_mine if USE_MINE else H_cv

# 체인: 이웃 H를 곱해 '모든 영상 → 기준 영상' 변환을 만든다.
#   기준 왼쪽  i : (i→i+1) 을 REF까지 누적      H_i = H_{i+1} · H_(i,i+1)
#   기준 오른쪽 i : (i-1→i) 의 역행렬을 누적     H_i = H_{i-1} · H_(i-1,i)^-1
H_use = {REF: np.eye(3)}
for i in range(REF - 1, -1, -1):
    H = H_use[i + 1] @ H_adj[(i, i + 1)];              H_use[i] = H / H[2, 2]
for i in range(REF + 1, len(imgs)):
    H = H_use[i - 1] @ np.linalg.inv(H_adj[(i - 1, i)]); H_use[i] = H / H[2, 2]

print(f'\n이후 워핑에는 {"내 DLT" if USE_MINE else "OpenCV"}의 H를 사용합니다.')
print(f'체인 연결 완료 — {len(imgs)}장 모두 기준 영상 {REF} 좌표계로 (곱한 H 개수: '
      + ', '.join(f'{i}:{abs(i - REF)}' for i in range(len(imgs))) + ')')


In [ ]:
# ── 셀 7 · 인라이어 진단 — 개수보다 배치 ────────────────────────
fig, axes = plt.subplots(1, len(matches), figsize=(5.6 * len(matches), 3.6))
axes = np.atleast_1d(axes)
for ax, ((i, j), (src, dst, _)) in zip(axes, matches.items()):
    m = inl_masks[(i, j)]
    ax.scatter(dst[~m, 0], dst[~m, 1], s=8, c='#C0392B', label='아웃라이어')
    ax.scatter(dst[m, 0],  dst[m, 1],  s=8, c='#2E9E4F', label='인라이어')
    ax.set_xlim(0, imgs[j].shape[1]); ax.set_ylim(imgs[j].shape[0], 0)
    ax.set_title(f'영상 {j} 위 인라이어 분포 ({i}→{j})'); ax.legend(loc='lower right', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.savefig(OUT / 'inlier_layout.png', dpi=120); plt.show()
print('겹침 영역 전체에 퍼져 있으면 양호 — 군집·공선이면 H 불안정 (어제 점 배치 교훈).')
print('체인에서는 한 쌍만 나빠도 그 바깥 영상 전체가 흔들립니다 — 약한 쌍이 있는지 확인하세요.')


### Step 4 · 캔버스 계산과 역방향 워핑

기준 영상은 그대로, 나머지는 $T\!\cdot\!H_i$ 로 워핑합니다. 여기서 $H_i$ 는 셀 6에서 체인으로
만든 **영상 $i$ → 기준 영상** 변환입니다 (기준에서 $k$ 칸 떨어진 영상은 이웃 H를 $k$ 번 곱한 것).
`warpPerspective`의 내부는 **역방향 매핑** — 어제 데모의 그 원리입니다.

> **평면 호모그래피의 한계** — 총 회전이 커질수록 바깥 영상이 급격히 늘어납니다(투영이 $\tan$ 처럼
> 발산). 6장을 한 번에 이으면 캔버스가 수천만~수억 화소로 폭주해 워핑·블렌딩이 멈춥니다.
> 그래서 아래 셀은 `MAX_CANVAS_MP` 상한을 두고, 넘으면 **기준에서 먼 영상부터 제외**하며
> 무엇을 뺐는지 출력합니다. 전부 이으려면 **원통 투영**(테이크홈 B)이 정답이고,
> 촬영 단계에서는 **3~4장 · 총 회전 90° 이내**를 권장합니다.

In [ ]:
# ── 셀 8 · 캔버스 + 워핑 (N장, 캔버스 상한 가드) ────────────────
MAX_CANVAS_MP = 40.0        # 캔버스 상한(백만 화소). 넘으면 기준에서 먼 영상부터 제외

def img_corners(im):
    h, w = im.shape[:2]
    return np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)

def canvas_of(idx):
    pts = np.vstack([cv2.perspectiveTransform(img_corners(imgs[i]), H_use[i])
                     for i in idx]).reshape(-1, 2)
    a = np.floor(pts.min(0)).astype(int); b = np.ceil(pts.max(0)).astype(int)
    return int(a[0]), int(a[1]), int(b[0] - a[0]), int(b[1] - a[1])

# 평면 호모그래피는 시야각이 넓어질수록 바깥 영상을 급격히 늘립니다(tan 발산).
# 총 회전이 크면 캔버스가 수억 화소로 폭주해 워핑·블렌딩이 사실상 멈추므로,
# 상한을 넘으면 기준에서 먼 영상부터 제외합니다 — 무엇을 왜 뺐는지 반드시 출력합니다.
use, dropped = list(range(len(imgs))), []
while True:
    x0, y0, CW, CH = canvas_of(use)
    if CW * CH / 1e6 <= MAX_CANVAS_MP or len(use) <= 2:
        break
    far = max(use, key=lambda i: abs(i - REF))
    print(f'⚠ 캔버스 {CW}x{CH} ({CW*CH/1e6:.0f}MP) > 상한 {MAX_CANVAS_MP:.0f}MP '
          f'→ 영상 {far} 제외 (기준 {REF}에서 {abs(far - REF)}칸)')
    dropped.append(far); use.remove(far)
if dropped:
    print(f'   제외: {sorted(dropped)} — 넓은 회전은 평면 H의 한계입니다.')
    print(f'   대안: 촬영을 3~4장(총 회전 90° 이내)으로 나누기 · 원통 투영(테이크홈 B) ·')
    print(f'         MAX_CANVAS_MP를 올리기(메모리·시간 급증 주의)')

T = np.array([[1, 0, -x0], [0, 1, -y0], [0, 0, 1]], float)
print(f'캔버스 {CW} x {CH} ({CW*CH/1e6:.1f}MP) · 사용 {len(use)}/{len(imgs)}장 {use}')

warped, masks = {}, {}
for i in use:
    Hf = T @ H_use[i]                     # 기준 영상은 H_use[REF] = I 이라 평행이동만
    warped[i] = cv2.warpPerspective(imgs[i], Hf, (CW, CH))
    masks[i] = cv2.warpPerspective(np.ones(imgs[i].shape[:2], np.uint8) * 255, Hf, (CW, CH))
    cols = np.where((masks[i] > 0).any(0))[0]
    print(f'  영상 {i}: 캔버스 점유 {(masks[i] > 0).mean()*100:4.1f}% | 가로 {cols.min()}~{cols.max()}')

acc = np.zeros((CH, CW, 3), np.float32); cnt = np.zeros((CH, CW, 1), np.float32)
for i in use:
    mm = (masks[i] > 0)[..., None].astype(np.float32)
    acc += warped[i].astype(np.float32) * mm; cnt += mm
overlap_vis = (acc / np.maximum(cnt, 1)).astype(np.uint8)
show(overlap_vis, '정합 확인(단순 평균) — 겹침 영역의 구조물이 거의 일치하면 통과 (살짝 어긋남 = 정상)',
     (11, 5.2))


---
### Step 5 · 알파 블렌딩 — `빈칸 ③`

경계에서 뚝 끊기는 이음새를, 각 영상의 **경계로부터의 거리**(distance transform)를 가중치로 써서
부드럽게 섞습니다. 겹침 구간에서 두 가중치의 합은 1이어야 합니다.

In [ ]:
# ── 셀 9 · 블렌딩 (빈칸 ③) → 내 파노라마 완성 ───────────────────
def second_weight(w1):
    # TODO ③: 영상 2의 가중치를 w1으로부터 계산해 반환 (한 줄)
    return None

_t = second_weight(np.float32([0.3]))
if _t is None:
    from solutions import second_weight
    print('⚠ 빈칸 ③ 미완성 — solutions로 임시 진행 (채운 뒤 이 셀부터 재실행)')
else:
    assert abs(float(_t[0]) + 0.3 - 1.0) < 1e-6, '빈칸 ③: 두 가중치의 합이 1이 되어야 합니다'
    print('빈칸 ③ 자기 구현 사용 ✅')

def feather(mask):
    d = cv2.distanceTransform((mask > 0).astype(np.uint8), cv2.DIST_L2, 3)
    return d / (d.max() + 1e-9)

# 기준 영상에서 시작해 가까운 이웃부터 바깥으로 한 장씩 누적한다 (N장 모두).
order = sorted((i for i in use if i != REF), key=lambda i: abs(i - REF))
acc = warped[REF].astype(np.float32)
acc_w = feather(masks[REF])
for i in order:
    w_i = feather(masks[i])
    both = (acc_w > 0) & (w_i > 0)
    w1 = np.where(both, acc_w / (acc_w + w_i + 1e-12), (acc_w > 0).astype(np.float32))
    w2 = np.where(both, second_weight(w1), (w_i > 0).astype(np.float32))
    acc = acc * w1[..., None] + warped[i].astype(np.float32) * w2[..., None]
    acc_w = np.maximum(acc_w, w_i)
pano = np.clip(acc, 0, 255).astype(np.uint8)
cv2.imwrite(str(OUT / 'my_panorama.jpg'), pano)
show(pano, f'내 파노라마 — {SRC_NOTE} {len(use)}장'
     + (' · 왜곡 보정 사용' if UNDISTORT else ''), (12, 5.6))
print(f'합성 순서: {REF}(기준) → ' + ' → '.join(map(str, order)))
print('저장 → out/my_panorama.jpg')


In [ ]:
# ── 셀 10 · 고스트 관찰 — 블렌딩은 기하를 못 고친다 ──────────────
i0 = min((i for i in use if i != REF), key=lambda i: abs(i - REF))
both = (masks[REF] > 0) & (masks[i0] > 0)
ys, xs = np.where(both)
if len(xs):
    cx, cy = int(xs.mean()), int(ys.mean()); hw, hh = 260, 200
    crop = pano[max(0, cy-hh):cy+hh, max(0, cx-hw):cx+hw]
    show(crop, f'영상 {REF}↔{i0} 겹침 확대 — 이중상(고스트)이 보이면 그 원인은?', (7.5, 5.2))
print('관찰 포인트')
print(' · 고스트가 안 보인다 → 제자리 회전이 잘 지켜졌고(시차↓) H·블렌딩이 제 역할')
print(' · 가까운 물체 주변만 이중상 → 손떨림 병진의 시차 — 오전 반례의 실물')
print(' · 바깥쪽 영상일수록 어긋남이 커진다 → 체인의 누적 오차 (H를 여러 번 곱한 결과)')
print(' · UNDISTORT를 False로 바꿔 셀 3부터 재실행 → 이음새 품질 비교 (내 캘리브의 기여 확인)')


### 선택 구현 (테이크홈) — 이해를 확장하고 싶다면

- **A. RANSAC 직접 구현** — `my_ransac(src, dst, estimate_H_dlt)` 를 처음부터. 골격·참조는 `solutions.py`.
  자기 RANSAC + 자기 DLT 조합이 `cv2.findHomography`와 인라이어 수·오차에서 동급이면 성공.
- **B. 원통 투영** — 3장 이상 넓은 파노라마에서 가장자리 늘어짐 완화. `solutions.cylindrical_warp(img, f)`
  — 내 캘리브의 **fx**를 그대로 f로 사용하는 것이 포인트 (K의 재사용).
- **C. 체인의 누적 오차** — `my_pano/`에 4~5장을 넣고, 기준(`REF`)을 끝(0)으로 바꿔 실행해 보세요.
  H를 여러 번 곱할수록 바깥 영상이 휘어집니다 — 기준을 가운데 두는 이유가 여기서 눈에 보입니다.
- **D. 시간 정합 다듬기** — 파트 3(보너스)에서 `corr_picker`로 15점을 더 정밀하게 재클릭,
  또는 `solutions.refine_dst_local` 자동 정제.

---
## 랩업 — 오늘의 결론

1. **내 폰도 캘리브레이션 대상** — 규칙(1× 고정·AF/AE 락)만 지키면 fx±σ까지 얻는다 *(파트 1)*
2. 순수 회전 → **H 하나로 정합** — 이론 3의 약속을 내 사진으로 검증
3. **DLT를 손으로** — A 두 행이 전부였고, 정규화가 그것을 실전급으로 만든다
4. RANSAC은 오매칭 속에서 모델을 구출 — 인라이어는 **개수보다 배치**
5. 블렌딩은 밝기를 섞을 뿐 — 어긋남(시차·H 오차)은 고스트로 폭로된다


In [ ]:
# ── 셀 12 · 결과 요약 저장 ──────────────────────────────────────
summary = dict(
    source=SRC_NOTE, undistort=bool(UNDISTORT), ratio=float(RATIO), ref_index=int(REF),
    n_images=len(imgs), files=[f.name for f in pano_files],
    used_images=list(use), dropped_images=sorted(dropped),
    proc_size=[int(PW), int(PH)], raw_size=[int(RW), int(RH)], scale=float(SCALE),
    pairs={f'{i}->{j}': dict(n_match=int(len(matches[(i, j)][2])),
                             n_inlier=int(inl_masks[(i, j)].sum()),
                             H_mine=H_mine[(i, j)].tolist(),
                             H_cv=H_cv[(i, j)].tolist()) for (i, j) in matches},
    H_to_ref={str(i): H_use[i].tolist() for i in range(len(imgs))},
    my_calib=None if not MY else dict(rms=MY['rms'], fx=float(MY['K'][0, 0]),
                                      sigma_fx=MY['sigma_fx'], n=MY['n']),
)
json.dump(summary, open(OUT / 'results.json', 'w'), indent=2)
print('저장 완료 →', sorted(p.name for p in OUT.iterdir()))
